# Estudo Comparativo de Classificação Acústica Submarina — Dataset IARA
### Notebook 2: Otimização com Opção de Rejeição (Filtro de Confiança)

Este notebook avalia a incorporação de um **operador de decisão com opção de rejeição** baseado em concordância temporal de janelas. Analisamos a curva de trade-off entre a taxa de cobertura (fração de áudios classificados) e as métricas de acurácia/SP à medida que exigimos maior certeza nas decisões.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

### 1. Estruturação dos Dados de Rejeição (LOFAR SVM vs MEL SVM vs CNN vs MLP)
Consolidamos as métricas sob os limiares críticos: $t=0$ (sem rejeição), $t \ge 0.5$ (maioria simples), $t \ge 0.6$ (maioria absoluta) e $t \ge 0.9$ (consenso crítico tático).

In [ ]:
rejection_data = {
    'Threshold': [0.0, 0.5, 0.6, 0.9],
    'LOFAR_SVM_Cov': [100.0, 78.25, 61.21, 22.27],
    'LOFAR_SVM_ACC': [63.32, 70.36, 76.54, 90.43],
    'MEL_SVM_Cov': [100.0, 89.67, 76.07, 37.48],
    'MEL_SVM_ACC': [64.56, 66.23, 69.60, 83.42],
    'CNN_Cov': [100.0, 95.93, 82.90, 54.01],
    'CNN_ACC': [65.01, 63.18, 66.61, 74.39],
    'MLP_Cov': [100.0, 85.60, 70.36, 32.90],
    'MLP_ACC': [63.54, 67.41, 71.97, 85.93]
}

df_rej = pd.DataFrame(rejection_data)
df_rej

### 2. Plotagem da Curva de Trade-off Cobertura-Acurácia
Um gráfico clássico de curva de trade-off onde o eixo X representa a Cobertura (%) e o eixo Y representa a Acurácia (%). Os modelos mais robustos se situam no canto superior direito (alta cobertura e alta acurácia).

In [ ]:
plt.figure(figsize=(10, 7))

# Plotagem de cada curva de modelo
plt.plot(df_rej['LOFAR_SVM_Cov'], df_rej['LOFAR_SVM_ACC'], 'o-', label='LOFAR SVM (Ours)', color='#e74c3c', linewidth=2.5, markersize=8)
plt.plot(df_rej['MEL_SVM_Cov'], df_rej['MEL_SVM_ACC'], 's-', label='MEL SVM (Ours)', color='#2ecc71', linewidth=2.0, markersize=8)
plt.plot(df_rej['CNN_Cov'], df_rej['CNN_ACC'], '^--', label='CNN Mel (Baseline)', color='#3498db', linewidth=2.0, markersize=8)
plt.plot(df_rej['MLP_Cov'], df_rej['MLP_ACC'], 'd-.', label='MLP Mel (Baseline)', color='#9b59b6', linewidth=2.0, markersize=8)

# Anotações de texto para os limiares nos pontos do LOFAR SVM
for idx, row in df_rej.iterrows():
    plt.annotate(f"t={row['Threshold']}", 
                 (row['LOFAR_SVM_Cov'], row['LOFAR_SVM_ACC']),
                 textcoords="offset points", 
                 xytext=(10,-10), 
                 ha='center', fontsize=9, color='#c0392b', weight='bold')

plt.title('Curva de Trade-off Cobertura-Acurácia sob Opção de Rejeição', fontsize=14, weight='bold')
plt.xlabel('Taxa de Cobertura de Classificação (%)', fontsize=12)
plt.ylabel('Acurácia Global de Teste (%)', fontsize=12)
plt.xlim(15, 105)
plt.ylim(60, 95)
plt.legend(fontsize=11, loc='upper right')
plt.tight_layout()
plt.show()

### 3. Discussão da Física do Gráfico:
1. **O Sniper de Picos (LOFAR SVM):** Sob limiar severo ($t \ge 0.9$), o SVM LOFAR atinge impressionantes **90.43% de acurácia** (o recorde geral do projeto). Ele faz isso descartando janelas ruidosas e focando apenas nas harmônicas nítidas do motor.
2. **A Superconfiança da CNN:** Sob $t \ge 0.9$, a CNN mantém uma cobertura alta (**54.01%**), porém a sua acurácia satura em medíocres **74.39%**. Convoluções profundas geram ativações Softmax saturadas (superconfiantes) que forçam decisões erradas de maioria temporal em sinais ruidosos. O SVM, por trabalhar com margens rígidas, calibra a incerteza de forma muito mais honesta.